# Step 2: Extract contract terms with GPT-5

**Goal:** turn page text into structured, cited terms you can review.

The flow is **PDF -> page text -> OpenAI -> Pydantic model -> citation checks -> review**.
We will first practice without an API call, then extract from one real contract.
Keep reusable code in `electricity_optimizer/` and use this notebook to explore it.
Select the project's `.venv` kernel and run cells in order with **Shift+Enter**.

Unlike Step 1, the live extraction sends selected document text to OpenAI and uses
your API account. The practice cells stay local. No cost comparison happens yet.

## 1. Import the building blocks
`AgreementTerms` is our output schema. `extract_agreement` calls OpenAI and returns
an extraction record. `validate_terms` checks evidence locally.

In [1]:
from pathlib import Path
import os
import json

from dotenv import load_dotenv
from electricity_optimizer.ingestion import read_pdf
from electricity_optimizer.models import PDFDocument, PDFPage
from electricity_optimizer.agreements import AgreementTerms, ContractTerm, Citation, ExtractionRecord
from electricity_optimizer.extraction import (
    build_document_input, extract_agreement, validate_terms, ExtractionError,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "contracts").is_dir():
    raise RuntimeError("Run this notebook from the project root.")
print("Project:", PROJECT_ROOT)

Project: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer


## 2. Understand the schema
Each field has a `status`, a `value`, and `citations` (page number plus exact quote).
- `found`: supported by the supplied document.
- `not_stated`: unknown; value is `None` and citations are empty.
- `ambiguous`: unclear or conflicting; preserve the uncertainty and evidence.

A missing fee is **not** a zero fee. Values retain currency, units, conditions and
exceptions as text. A later stage will convert reviewed pricing terms into numeric rules.
Notice that `energy_charges` and `average_prices` are separate fields.

In [2]:
list(AgreementTerms.model_fields)

['provider',
 'plan_name',
 'market',
 'customer_type',
 'document_type',
 'effective_date',
 'rate_type',
 'energy_charges',
 'average_prices',
 'recurring_charges',
 'delivery_charges',
 'credits_and_minimum_usage',
 'contract_length',
 'termination_fee',
 'renewal_terms',
 'price_change_rules']

## 3. Practice locally with invented text
This tiny synthetic example is explicitly **not** a real provider agreement or AI output.
We fill one field manually to understand what the model will return.
All other fields remain unknown.

In [3]:
practice_document = PDFDocument(
    source_file="SYNTHETIC_PRACTICE_ONLY",
    sha256="not-a-real-file-fingerprint",
    pages=[PDFPage(page_number=1, text="Contract length: 12 months.", needs_review=False)],
)
practice_terms = AgreementTerms(**{
    name: ContractTerm(status="not_stated", value=None, citations=[])
    for name in AgreementTerms.model_fields
})
practice_terms.contract_length = ContractTerm(
    status="found", value="12 months",
    citations=[Citation(page_number=1, quote="Contract length: 12 months.")],
)
practice_terms.contract_length.model_dump()

{'status': 'found',
 'value': '12 months',
 'citations': [{'page_number': 1, 'quote': 'Contract length: 12 months.'}]}

## 4. Try the evidence checker
The first check should find no errors for our cited term. Unknown fields still generate
warnings. Then deliberately invent a quote and see the validator reject it.
Changing only the value while leaving a real quote can escape this check: semantic
accuracy and completeness still need review.

In [4]:
practice_issues = validate_terms(practice_terms, practice_document)
print("Practice errors:", sum(i.severity == "error" for i in practice_issues))
print("Practice warnings:", sum(i.severity == "warning" for i in practice_issues))

bad_terms = practice_terms.model_copy(deep=True)
bad_terms.contract_length.citations[0].quote = "Contract length: 36 months."
for issue in validate_terms(bad_terms, practice_document):
    if issue.severity == "error":
        print(issue.field, issue.message)

Practice errors: 0
Practice warnings: 15
contract_length Quote not found on page 1.


## 5. Select a real PDF and preview the input
Start with the one-page Reliant EFL. Change `selected_path` to another contract later.
All pages of the selected PDF are sent with one-based page numbers. No other files
are included. The lesson rejects oversized documents instead of truncating pages.

In [10]:
selected_path = PROJECT_ROOT / "contracts" / "R1F00160000021A.pdf"
document = read_pdf(selected_path)
payload = build_document_input(document)
print("Selected:", document.source_file)
print("Pages:", len(document.pages))
print("Input characters (not tokens):", len(payload))
print(payload[:1200])

Selected: R1F00160000021A.pdf
Pages: 2
Input characters (not tokens): 5026
{"source_file": "R1F00160000021A.pdf", "pages": [{"page_number": 1, "text": "Electricity Facts Label (EFL)\n                             Reliant Energy Retail Services, LLC\n                          Reliant Power Savings 2,000 kWh 24 plan\n                     AEP Texas Central service area\n                                    Date: 09/01/2026\n                        Average monthly use:      500 kWh         1000 kWh         2000 kWh\n                      Average price per kWh:         21.8 ¢              21.5 ¢              13.8 ¢\n\n             The price you pay each month consists of the Base Charge, Energy Charge, Usage Credit and TDSP Pass-\n             Through Charges in effect for your monthly billing cycle. A Usage Credit of $150.00 will be included for\n              each billing cycle when your usage on this plan is above or equal to 2,000 kWh. There is no Usage Credit\n                 for a bill

## 6. Configure your API key locally
Copy `.env.example` to `.env` in the project root. Set `OPENAI_API_KEY` there and
leave `OPENAI_MODEL=gpt-5`. Do not put the key in notebook cells or outputs.
The `.env` file is excluded from Git. This cell displays only whether a key is present.
If you change an already loaded key, restart the kernel before rerunning.

If dependencies are missing, run this in a terminal and restart the kernel:
`.\.venv\Scripts\python.exe -m pip install -r requirements-notebook.txt`

In [11]:
load_dotenv(PROJECT_ROOT / ".env", override=False)
model = os.getenv("OPENAI_MODEL") or "gpt-5"
print("Model:", model)
print("API key configured:", bool(os.getenv("OPENAI_API_KEY", "").strip()))

Model: gpt-5
API key configured: True


## 7. Run one live extraction
Set the switch below to `True` when ready to make the API call. Re-running this cell
makes another request; there is no automatic cache or retry. Leave it `False` to
run the lesson locally. The model may take a minute or two.

`responses.parse(..., text_format=AgreementTerms)` asks for structured output and
parses it into our Pydantic schema. A refusal or incomplete response is an error,
not an empty successful extraction. `store=False` disables response storage but
is not a promise of zero retention under all API policies.

In [12]:
RUN_LIVE_EXTRACTION = True
record = None  # Clear previous state so a failed call cannot look like a new success.
if RUN_LIVE_EXTRACTION:
    try:
        record = extract_agreement(document, model=model)
        print("Extracted:", record.source_file)
        print("Review status:", record.review_status)
    except ExtractionError as error:
        print(str(error))
else:
    print("Live extraction skipped. Set RUN_LIVE_EXTRACTION = True to send the selected document.")

Extracted: R1F00160000021A.pdf
Review status: needs_human_review


## 8. Review the extracted terms and citations
Check the energy charge, average prices, contract length and termination conditions
against the original PDF. Review all errors and warnings before using terms.
Even a result with zero issues remains `needs_human_review`.
We have not yet checked plan compatibility or pricing completeness.

In [13]:
if record is None:
    print("No live result yet. Run the previous cell with a configured key.")
else:
    for name in AgreementTerms.model_fields:
        term = getattr(record.terms, name)
        print(f"\n{name} [{term.status}]: {term.value}")
        for citation in term.citations:
            print(f"  Page {citation.page_number}: {citation.quote}")
    print("\nREVIEW ISSUES")
    for issue in record.issues:
        print(f"{issue.severity.upper()} | {issue.field}: {issue.message}")


provider [found]: Reliant Energy Retail Services, LLC
  Page 1: Reliant Energy Retail Services, LLC

plan_name [found]: Reliant Power Savings 2,000 kWh 24 plan
  Page 1: Reliant Power Savings 2,000 kWh 24 plan

market [found]: AEP Texas Central service area
  Page 1: AEP Texas Central service area

customer_type [not_stated]: None

document_type [found]: Electricity Facts Label (EFL)
  Page 1: Electricity Facts Label (EFL)

effective_date [found]: Date: 09/01/2026
  Page 1: Date: 09/01/2026

rate_type [found]: Fixed Rate
  Page 1: Type of Product                           Fixed Rate

energy_charges [found]: Energy Charge: 15.4124¢ per kWh
  Page 1: Energy Charge:  15.4124¢   per kWh

average_prices [found]: Average price per kWh at average monthly use levels: 21.8 ¢ at 500 kWh; 21.5 ¢ at 1000 kWh; 13.8 ¢ at 2000 kWh.
  Page 1: Average monthly use:      500 kWh         1000 kWh         2000 kWh
  Page 1: Average price per kWh:         21.8 ¢              21.5 ¢              13.8 ¢

rec

## 9. Save the live result for review
The saved record includes the source fingerprint, model, response ID, timestamp,
prompt version, terms and validation issues. Practice data is not exported here.
Each response uses its own filename; re-saving the same response overwrites that file.

In [14]:
if record is None:
    print("Nothing to save: no live extraction has completed.")
elif record.source_sha256 != document.sha256 or record.source_file != document.source_file:
    raise RuntimeError("Selected document changed. Rerun the live extraction before saving.")
else:
    output_dir = PROJECT_ROOT / "output" / "extraction"
    output_dir.mkdir(parents=True, exist_ok=True)
    # Keep only filename-safe characters in the API response identifier.
    safe_id = "".join(c for c in record.response_id if c.isalnum() or c in "-_")
    output_path = output_dir / f"{selected_path.stem}_{safe_id}.json"
    output_path.write_text(record.model_dump_json(indent=2), encoding="utf-8")
    restored = ExtractionRecord.model_validate_json(output_path.read_text(encoding="utf-8"))
    assert restored == record
    print("Saved review record:", output_path)

Saved review record: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\extraction\R1F00160000021A_resp_013d02c2f25ef043016aa47d79cf8487d185d1da3a719914e7.json


## What changed from Step 1?
PyMuPDF still reads the pages. GPT-5 now interprets them, Pydantic constrains the
result's structure, and Python checks the quotes. None of these alone guarantees
that a contract has been interpreted correctly.

**Exercise:** inspect `delivery_charges` for the Reliant PDF. Does it contain a
numerical amount or a pass-through condition? Why would a pricing engine need that
distinction? Then inspect `average_prices` versus `energy_charges`.

**Next:** a LangGraph supervisor will coordinate extraction and validation as
explicit steps. Usage ingestion and deterministic cost calculation come afterward.

References: [OpenAI structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
and [GPT-5](https://developers.openai.com/api/docs/models/gpt-5).
Restart the kernel after editing imported Python modules. Clear outputs before sharing private data.